# 03. Monolingual, Bilingual & Lexical Resource Statistics
How much training data does each experiment (E1-E7) actually have available, per translation direction?

In [ ]:
# ============================================================
# PATH BOOSTER — Works on Kineses / Jupyter / Colab / Kaggle
# Handles FileNotFoundError when kernel CWD no longer exists.
# ============================================================
import os, sys

# Step 1: Safely get current directory
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

# Step 2: Navigate to project root (Kineses home-based path)
kineses_proj = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(kineses_proj):
    os.chdir(kineses_proj)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Step 3: Add project root to Python path
proj_root = os.getcwd()
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [ ]:
from src.master_corpus.manager import MasterCorpusManager
from src.task_generation.translation_pairs import direction_counts

manager = MasterCorpusManager()
train_df = manager.load_train_split()
counts = direction_counts(train_df)
for direction, count in counts:
    print(f'{direction:25s} {count:>8,} pairs')

## Complete triplets available for E4 (Trilingual)

In [ ]:
complete_triplets = train_df.dropna(subset=['English', 'Kiswahili', 'Ekegusii'])
print(f'Complete triplets: {len(complete_triplets):,} / {len(train_df):,} '
      f'({100 * len(complete_triplets) / len(train_df):.1f}%)')

## Lexical corpus resource size (E6)

In [ ]:
lexical_df = manager.load_lexical_corpus()
print(f'Lexical entries: {len(lexical_df):,}')
for lang in ['English', 'Kiswahili', 'Ekegusii']:
    non_null = lexical_df[lang].notna().sum()
    print(f'  {lang}: {non_null}/{len(lexical_df)} non-null ({100*non_null/len(lexical_df):.1f}%)')

## Resource size by experiment (approximate task counts)

In [ ]:
from src.experiments.bilingual import BilingualExperiment

e1_pairs = direction_counts(train_df)
eng_eke = sum(c for d, c in e1_pairs if d in ('English->Ekegusii', 'Ekegusii->English'))
swa_eke = sum(c for d, c in e1_pairs if d in ('Kiswahili->Ekegusii', 'Ekegusii->Kiswahili'))
print(f'E1 (English-Ekegusii) task pool: {eng_eke:,}')
print(f'E2 (Swahili-Ekegusii) task pool:  {swa_eke:,}')
print(f'E3 (Combined bilingual) task pool: {eng_eke + swa_eke:,}')